# Proteome-wide protein shape analysis

Reads `results/metrics.parquet` (one row per protein x unit x mask variant x domain) and runs the Stage 5 analyses from `protshape.analysis`. Classes and domain labels are UniProt-annotation based (`class_annotation`); `class_final` is a sensitivity analysis.

In [ ]:
import pandas as pd, numpy as np
from pathlib import Path
from IPython.display import Image, display
from protshape import analysis as A
pd.set_option('display.width', 200); pd.set_option('display.max_columns', 40)
import os
WORK = Path(os.environ.get('PROTSHAPE_WORK', '../results' if Path('../results/metrics.parquet').exists() else 'results'))
m = pd.read_parquet(WORK / 'metrics.parquet')
metrics = A.metric_columns(m[(m.unit == 'DOMAIN') & (m.mask_variant == 'default')])
print(m.accession.nunique(), 'proteins;', len(metrics), 'metrics')
A.domains(m).groupby(['taxid', 'group']).size().unstack(fill_value=0)

## 1. Size dependence (within SOLUBLE_DOMAIN / SOLUBLE proteins)

In [ ]:
sd = A.size_dependence(m, metrics)
sd[sd.unit == 'DOMAIN'].assign(a=lambda x: x.spearman_rho_vs_N.abs()).sort_values('a').drop(columns='a').round(3)

## 2. Membrane vs soluble domains, size-matched (Cliff's delta and AUC per size bin)

In [ ]:
mc = A.main_comparison(m, metrics)
key = A.SHAPE + A.HYDRO
mc[mc.metric.isin(key) & (mc.a == 'MEMBRANE_DOMAIN') & (mc.b == 'SOLUBLE_DOMAIN')].pivot_table(index='metric', columns='size_bin', values='auc').round(2)

Per organism (stratified AUC), then pooled with organism as a stratum:

In [ ]:
per = []
for org, t in ((9606, 'human'), (559292, 'yeast')):
    r = A.main_comparison(m, metrics, organism=org)
    per.append(r[(r.size_bin == 'stratified') & (r.a == 'MEMBRANE_DOMAIN') & (r.b == 'SOLUBLE_DOMAIN')][['metric', 'auc']].rename(columns={'auc': t}))
pooled = mc[(mc.size_bin == 'stratified') & (mc.a == 'MEMBRANE_DOMAIN') & (mc.b == 'SOLUBLE_DOMAIN')][['metric', 'auc']].rename(columns={'auc': 'pooled'})
per[0].merge(per[1], on='metric').merge(pooled, on='metric').set_index('metric').loc[lambda x: x.index.isin(key)].round(2)

## 3. Secondary-structure control (CATH class 1) and beta barrels

In [ ]:
ctrl = A.cath1_control(m, metrics)
A.surviving_metrics(mc, ctrl).round(3).head(25)

In [ ]:
A.barrel_vs_cath2(m, metrics).query("size_bin == 'stratified'").sort_values('auc').round(3).head(10)

## 4. Whole-protein view and where elongation lives

In [ ]:
A.elongation_location(m).round(3)

In [ ]:
wp = A.whole_protein(m, metrics)
wp[(wp.size_bin == 'stratified') & wp.metric.isin(A.SHAPE)].pivot_table(index='metric', columns='comparison', values='auc').round(2)

## 5. Prolate vs oblate

In [ ]:
A.prolateness_by_group(m).round(3)

## 6. Hydrophobicity vs depth, and recovering the membrane label from structure alone

In [ ]:
A.hydrophobicity_by_group(m).round(2)

In [ ]:
A.structure_only_roc(m, metrics).sort_values('auc_oriented', ascending=False).round(3).head(20)

## 7. Sensitivity: pLDDT 50 / 90, no TED exclusion, class_final

In [ ]:
sens = A.sensitivity(m, metrics)
sens[sens.conclusion_moves_with != ''].round(3)

## 8. What TED's globularity terms capture

In [ ]:
A.ted_globularity_correlation(m, metrics).assign(a=lambda x: x.spearman.abs()).sort_values('a', ascending=False).drop(columns='a').round(3).head(20)

## 9. PCA and UMAP of DOMAIN units

In [ ]:
emb = A.embeddings(m, metrics)
print('PCA explained variance:', np.round(emb.attrs['explained'], 3))
A.plot_embeddings(emb, WORK / 'analysis' / 'embeddings.png')
display(Image(WORK / 'analysis' / 'embeddings.png'))

In [ ]:
A.plot_distributions(m, WORK / 'analysis' / 'distributions.png')
display(Image(WORK / 'analysis' / 'distributions.png'))

## 10. findings.md

`protshape analyze` writes `results/findings.md` from the same functions.